# CV 4대 태스크 실습 — ②탐지 실습1: Haar Cascade 얼굴 탐지

### 셀 1 — 패키지 설치 및 한글 폰트 설정

In [ ]:
!pip install -q opencv-python-headless matplotlib numpy
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import abc
import os
import urllib.request
from dataclasses import dataclass, field
from typing import List

import cv2
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np

# matplotlib 기본 폰트에는 한글이 없어 그래프 제목이 깨진다. 나눔고딕 폰트를 등록한다.
nanum_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if os.path.exists(nanum_path):
    fm.fontManager.addfont(nanum_path)
    plt.rcParams["font.family"] = fm.FontProperties(fname=nanum_path).get_name()
plt.rcParams["axes.unicode_minus"] = False



### 셀 2 — [도메인 계층 / Domain Layer]

In [ ]:
# ----------------------------------------------------------------------------
# 이 계층은 "탐지(Detection)"라는 개념 자체를 cv2 등 특정 구현 기술과 무관하게
# 정의한다. 값 객체(Value Object)와 추상 인터페이스(Port)만 존재하며,
# cv2를 import하거나 cv2 구체 타입을 사용하지 않는다 (DIP: 상위 계층은
# 구체 구현이 아닌 추상화에 의존해야 한다).
# ===== [도메인 계층 / Domain Layer] =====

@dataclass
class DetectionBox:
    """탐지된 객체 하나를 나타내는 불변 값 객체(Value Object).

    x, y, w, h: 바운딩 박스 좌표(좌상단 기준) 및 크기
    label: 탐지된 객체의 종류 (예: "face")
    confidence: 탐지 신뢰도(0~1). Haar Cascade처럼 신뢰도를 제공하지 않는
        알고리즘은 1.0으로 채워 인터페이스 형태를 동일하게 유지한다(LSP).
    """

    x: int
    y: int
    w: int
    h: int
    label: str = "object"
    confidence: float = 1.0


class ObjectDetector(abc.ABC):
    """객체 탐지기의 추상 인터페이스(Port).

    ISP(인터페이스 분리 원칙)에 따라 "탐지"라는 단일 책임만 가지며,
    시각화나 후처리(NMS) 책임은 포함하지 않는다. Haar Cascade, HOG, 추후
    추가될 DNN 기반 탐지기 등 모든 구체 구현은 이 인터페이스를 만족해야
    하며(LSP), 유스케이스 계층은 오직 이 추상화에만 의존한다(DIP).
    """

    @abc.abstractmethod
    def detect(self, image: np.ndarray) -> List[DetectionBox]:
        """입력 이미지(BGR ndarray)에서 객체를 탐지해 DetectionBox 리스트로 반환한다."""
        raise NotImplementedError



### 셀 3 — [유스케이스 계층 / Use Case Layer]

In [ ]:
# ----------------------------------------------------------------------------
# 애플리케이션의 흐름(탐지 실행 → 결과 집계)을 조율하는 계층. 구체적인 cv2
# 호출은 전혀 하지 않고, 생성자로 주입받은 ObjectDetector 추상화만 사용한다
# (DIP: 생성자 주입). SRP에 따라 "탐지 흐름 조율"이라는 책임만 가지며,
# 화면 출력/파일 저장 등은 구성 루트(Composition Root) 몫이다.
# ===== [유스케이스 계층 / Use Case Layer] =====

class DetectionUseCase:
    """객체 탐지 유스케이스.

    생성자 주입(Constructor Injection)으로 ObjectDetector 추상화를 전달받아
    detect()를 호출하고 결과를 그대로 반환한다. Haar Cascade를 HOG나 DNN
    탐지기로 교체하더라도 이 클래스는 전혀 수정할 필요가 없다(OCP).
    """

    def __init__(self, detector: ObjectDetector):
        self._detector = detector

    def run(self, image: np.ndarray) -> List[DetectionBox]:
        """주입된 탐지기로 이미지를 탐지하고 결과 목록을 반환한다."""
        return self._detector.detect(image)



### 셀 4 — [인프라/어댑터 계층 / Infrastructure Layer]

In [ ]:
# ----------------------------------------------------------------------------
# cv2(OpenCV)를 실제로 호출하는 유일한 계층. 도메인 인터페이스
# (ObjectDetector)를 구현하는 구체 어댑터 클래스를 정의한다. 이 계층만
# cv2 API에 의존하므로, OpenCV를 다른 라이브러리로 바꾸더라도 영향 범위가
# 이 파일의 이 부분으로 한정된다.
# ===== [인프라/어댑터 계층 / Infrastructure Layer] =====

class HaarCascadeFaceDetector(ObjectDetector):
    """Haar Cascade 분류기 기반 얼굴 탐지 어댑터.

    OpenCV는 사전 학습된 Haar Cascade XML 파일을 패키지에 내장하고 있다.
    ObjectDetector 인터페이스를 구현하므로, 유스케이스 계층 입장에서는
    다른 어떤 ObjectDetector 구현체와도 동일하게 취급된다(LSP).
    """

    def __init__(self, cascade_path: str = None,
                 scale_factor: float = 1.1, min_neighbors: int = 5,
                 min_size=(30, 30)):
        cascade_path = cascade_path or (cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
        self._cascade = cv2.CascadeClassifier(cascade_path)
        if self._cascade.empty():
            raise RuntimeError("Haar Cascade 파일을 불러오지 못했습니다: " + cascade_path)
        self._scale_factor = scale_factor
        self._min_neighbors = min_neighbors
        self._min_size = min_size

    def detect(self, image: np.ndarray) -> List[DetectionBox]:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        gray = cv2.equalizeHist(gray)  # 명암 대비를 균일화해 탐지율을 높인다

        # detectMultiScale: 여러 스케일(크기)로 이미지를 훑으며 얼굴 패턴을 탐색한다
        #  - scaleFactor: 매 스케일마다 이미지를 얼마나 축소할지 (1.1 = 10%씩 축소)
        #  - minNeighbors: 후보 영역이 몇 번 이상 겹쳐야 '진짜 얼굴'로 인정할지
        faces = self._cascade.detectMultiScale(
            gray, scaleFactor=self._scale_factor,
            minNeighbors=self._min_neighbors, minSize=self._min_size,
        )
        return [DetectionBox(int(x), int(y), int(w), int(h), label="face", confidence=1.0)
                for (x, y, w, h) in faces]


class DetectionVisualizer:
    """탐지 결과를 이미지 위에 그리는 책임만 담당하는 클래스(SRP).

    탐지 로직과 시각화 로직을 한 클래스에 섞지 않음으로써, 탐지기를
    교체하거나 시각화 스타일을 바꿀 때 서로 영향을 주지 않도록 분리했다.
    """

    def __init__(self, color=(0, 140, 255), thickness: int = 3):
        self._color = color
        self._thickness = thickness

    def draw(self, image: np.ndarray, boxes: List[DetectionBox]) -> np.ndarray:
        result = image.copy()
        for box in boxes:
            cv2.rectangle(result, (box.x, box.y), (box.x + box.w, box.y + box.h),
                           self._color, self._thickness)
            cv2.putText(
                result, box.label, (box.x, max(box.y - 10, 0)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, self._color, 2,
            )
        return result



### 셀 5 — [구성 루트 / Composition Root — Colab 실행 코드]

In [ ]:
# ----------------------------------------------------------------------------
# 구체 어댑터를 생성하고 유스케이스에 주입(DI)한 뒤, 실제 샘플 이미지에 대해
# 실행하고 결과를 matplotlib로 시각화/저장한다. 파일 I/O, 네트워크 다운로드,
# 플로팅은 이 계층에서만 수행한다.
# ===== [구성 루트 / Composition Root — Colab 실행 코드] =====

SAMPLE_PATH = "/content/people.jpg"
if not os.path.exists(SAMPLE_PATH):
    # 얼굴이 포함된 공개 테스트 이미지 (OpenCV 공식 샘플 저장소)
    url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg"
    urllib.request.urlretrieve(url, SAMPLE_PATH)
    print("샘플 이미지 다운로드 완료:", SAMPLE_PATH)

img_bgr = cv2.imread(SAMPLE_PATH)

# 구성 루트: 구체 어댑터(HaarCascadeFaceDetector)를 만들어 유스케이스에 주입한다.
detector = HaarCascadeFaceDetector()
use_case = DetectionUseCase(detector)
faces = use_case.run(img_bgr)
print(f"탐지된 얼굴 수: {len(faces)}")


### 셀 6 — 탐지 결과를 바운딩 박스로 시각화 (컴퓨터비전의 '탐지' 태스크 실습)

In [ ]:
visualizer = DetectionVisualizer()
img_result = visualizer.draw(img_bgr, faces)
img_result_rgb = cv2.cvtColor(img_result, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 6))
plt.imshow(img_result_rgb)
plt.title(f"얼굴 탐지 결과 ({len(faces)}개 탐지)")
plt.axis("off")
plt.savefig("/content/face_detection_result.png", dpi=120)
plt.show()
print("결과 저장: /content/face_detection_result.png")

# 참고: 본인 사진으로 테스트하려면 아래 코드로 업로드 후 SAMPLE_PATH를 바꾸면 된다.
# from google.colab import files
# uploaded = files.upload()
